## Cell 1 — Install dependencies

In [ ]:
# isko mat run karooooooooooooooooooooooooooooo!!!!!!
!pip install -q msclap                    # Microsoft CLAP
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q ffmpeg-python torchaudio librosa
!apt-get install -qq ffmpeg
print('✅ Installs done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 112.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 132.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 r

In [1]:
# Clean previous installs
!pip uninstall -y torch torchaudio torchvision

# Install CPU/GPU compatible PyTorch (CUDA 12.1 — stable)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Rest of your stack
!pip install -q numpy==1.26.4
!pip install -q transformers==4.41.2
!pip install -q msclap
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q ffmpeg-python librosa

# System dependency
!apt-get install -qq ffmpeg

print("✅ Clean CLAP environment ready")

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 96.1 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.5 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 68.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 33.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:0000:0100:01
  

In [2]:
import torch, torchaudio

print("Torch:", torch.__version__)
print("CUDA used by torch:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

Torch: 2.5.1+cu121
CUDA used by torch: 12.1
GPU available: True


## Cell 2 — Mount Drive & paths

In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE      = Path('/content/drive/MyDrive/videostory_prism')
VIDEO_DIR = BASE / 'videos/raw'
FEAT_DIR  = BASE / 'features'
CLAP_DIR  = FEAT_DIR / 'clap'       # new: CLAP audio event features
CLIP5_DIR = FEAT_DIR / 'clip5'      # new: 5-frame CLIP features
CLAP_DIR.mkdir(parents=True, exist_ok=True)
CLIP5_DIR.mkdir(parents=True, exist_ok=True)

videos = sorted(VIDEO_DIR.glob('*.mp4'))
print(f'Found {len(videos)} videos')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 4280 videos


## Cell 3 — CLAP model setup & sanity test

In [4]:
import torch
from msclap import CLAP

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Load CLAP (version 2023) — ~400MB download
clap_model = CLAP(version='2023', use_cuda=(DEVICE=='cuda'))

# Sanity test: embed a short sine wave
import numpy as np, torchaudio
sr = 44100
dummy_wav = torch.sin(2 * torch.pi * 440 * torch.arange(sr) / sr).numpy()
np.save('/tmp/test_sine.wav', dummy_wav)  # save as npy then test

# Test text embedding
text_emb = clap_model.get_text_embeddings(['car crash', 'dog barking', 'silence'])
print(f'CLAP text embedding shape: {text_emb.shape}')  # expect (3, 512)
print('✅ CLAP OK')

Device: cuda


CLAP_weights_2023.pth:   0%|          | 0.00/690M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

CLAP text embedding shape: torch.Size([3, 1024])
✅ CLAP OK


## Cell 4 — CLAP Audio Event Feature Extraction
Extracts 512-dim audio event embedding for each video.
Captures: crash sounds, tire screech, engine noise, glass breaking, music, speech.
**Run on high-VRAM machine (or T4 ~45 min)**

In [5]:
import subprocess, tempfile, os
import numpy as np, torch
from pathlib import Path
from tqdm.auto import tqdm

def extract_audio_wav(video_path, out_wav, sr=44100):
    """Extract mono audio from video using ffmpeg."""
    cmd = [
        'ffmpeg', '-y', '-i', str(video_path),
        '-ac', '1',               # mono
        '-ar', str(sr),           # sample rate
        '-vn',                    # no video
        str(out_wav)
    ]
    result = subprocess.run(cmd, capture_output=True)
    return result.returncode == 0

skipped, done, errors = 0, 0, 0
videos=videos[:350] #here
for vid_path in tqdm(videos):
    vid_id = vid_path.stem
    out_file = CLAP_DIR / f'{vid_id}.npy'
    if out_file.exists():
        skipped += 1
        continue

    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_wav = tmp.name

    try:
        ok = extract_audio_wav(vid_path, tmp_wav)
        if not ok or not os.path.exists(tmp_wav) or os.path.getsize(tmp_wav) < 100:
            # No audio track — store zero vector
            np.save(out_file, np.zeros(512, dtype=np.float32))
            skipped += 1
            continue

        # CLAP audio embedding
        audio_emb = clap_model.get_audio_embeddings([tmp_wav])  # (1, 512)
        emb = audio_emb[0].cpu().numpy().astype(np.float32)
        np.save(out_file, emb)
        done += 1

    except Exception as e:
        print(f'ERROR {vid_id}: {e}')
        np.save(out_file, np.zeros(512, dtype=np.float32))
        errors += 1
    finally:
        try: os.remove(tmp_wav)
        except: pass

print(f'\n✅ CLAP done | extracted={done} | skipped={skipped} | errors={errors}')

  0%|          | 0/350 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Cell 5 — Multi-Frame CLIP Extraction (5 temporal frames)
Extracts CLIP embeddings at 5 temporal positions → shape (5, 512).
Captures temporal changes: person leaving car, car driving away, smoke after crash.
**Runs on T4 — ~30 min for 350 videos**

In [ ]:
import clip, cv2, torch, numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, clip_preprocess = clip.load('ViT-B/32', device=DEVICE)
clip_model.eval()
print(f'CLIP loaded on {DEVICE}')

N_FRAMES = 5   # 0%, 25%, 50%, 75%, 100% of video

def extract_clip5(video_path):
    """Extract CLIP features at N_FRAMES positions → (N_FRAMES, 512)"""
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total = max(total, 1)

    # Compute frame indices
    positions = [int(total * p) for p in [0.0, 0.25, 0.50, 0.75, 0.99]]
    positions = [min(p, total - 1) for p in positions]

    frames = []
    for pos in positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ret, frame = cap.read()
        if not ret:
            # Fallback: duplicate previous frame or zeros
            frames.append(None)
        else:
            img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            frames.append(img)
    cap.release()

    # Replace None frames with first available frame
    first_valid = next((f for f in frames if f is not None), None)
    if first_valid is None:
        return np.zeros((N_FRAMES, 512), dtype=np.float32)
    frames = [f if f is not None else first_valid for f in frames]

    # CLIP encode
    imgs = torch.stack([clip_preprocess(f) for f in frames]).to(DEVICE)  # (5, 3, 224, 224)
    with torch.no_grad():
        feats = clip_model.encode_image(imgs)                             # (5, 512)
        feats = feats / feats.norm(dim=-1, keepdim=True)                  # L2-normalize
    return feats.cpu().numpy().astype(np.float32)

skipped, done, errors = 0, 0, 0

videos=videos[:350] #here
for vid_path in tqdm(videos):
    vid_id = vid_path.stem
    out_file = CLIP5_DIR / f'{vid_id}.npy'
    if out_file.exists():
        skipped += 1
        continue
    try:
        feat = extract_clip5(vid_path)    # (5, 512)
        np.save(out_file, feat)
        done += 1
    except Exception as e:
        print(f'ERROR {vid_id}: {e}')
        np.save(out_file, np.zeros((N_FRAMES, 512), dtype=np.float32))
        errors += 1

print(f'\n✅ CLIP5 done | extracted={done} | skipped={skipped} | errors={errors}')

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 134MiB/s]


CLIP loaded on cuda


  0%|          | 0/350 [00:00<?, ?it/s]

## Cell 6 — Validation

In [ ]:
import numpy as np
from pathlib import Path

clap_files  = list(CLAP_DIR.glob('*.npy'))
clip5_files = list(CLIP5_DIR.glob('*.npy'))

print(f'CLAP features   : {len(clap_files)} files')
print(f'CLIP5 features  : {len(clip5_files)} files')

# Sample check
sample_clap  = np.load(clap_files[0])
sample_clip5 = np.load(clip5_files[0])
print(f'CLAP shape  : {sample_clap.shape}   — expected (512,)')
print(f'CLIP5 shape : {sample_clip5.shape}  — expected (5, 512)')

# Zero-feature scan
n_zero_clap  = sum(1 for f in clap_files  if np.allclose(np.load(f), 0))
n_zero_clip5 = sum(1 for f in clip5_files if np.allclose(np.load(f), 0))
print(f'Zero CLAP  : {n_zero_clap}  ({100*n_zero_clap/len(clap_files):.1f}%)  — videos with no audio')
print(f'Zero CLIP5 : {n_zero_clip5} ({100*n_zero_clip5/len(clip5_files):.1f}%)  — should be 0')

# Audio event diversity: cosine similarity between 10 random CLAP embeddings
import random
sample_10 = random.sample(clap_files, min(10, len(clap_files)))
vecs = np.stack([np.load(f) for f in sample_10])
norms = np.linalg.norm(vecs, axis=1, keepdims=True)
normed = vecs / (norms + 1e-8)
cos_sim = (normed @ normed.T)
mask = ~np.eye(len(sample_10), dtype=bool)
print(f'\nCLAP avg cosine similarity (10 random videos): {cos_sim[mask].mean():.4f}')
print('  < 0.80 → Good diversity (CLAP distinguishes audio events)')
print('  > 0.95 → Low diversity (audio features not useful)')
print('\n✅ Validation complete. Copy CLAP_DIR and CLIP5_DIR paths to PRISM_BART_Colab')